# StereoCamera

A `StereoCamera` in GTSAM is composed of a `Pose3` for the left camera and a `Cal3_S2Stereo` calibration object. Its main function is to project a 3D point into the stereo image plane, which results in a `StereoPoint2` measurement, or to backproject a `StereoPoint2` into a 3D point. This class is essential for stereo vision applications where you need to relate 3D world points to 2D image measurements from a stereo rig.

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/geometry/doc/StereoCamera.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install --quiet gtsam-develop

In [1]:
import gtsam
import numpy as np
from gtsam import Cal3_S2Stereo, Point3, Pose3, Rot3, StereoCamera, StereoPoint2

## Initialization

A `StereoCamera` is initialized by providing the camera's pose in the world frame and its stereo calibration parameters. The pose, represented by a `Pose3` object, defines the position and orientation of the left camera. The calibration, represented by a `Cal3_S2Stereo` object, contains the intrinsic parameters of the cameras and the baseline between them.

In [2]:
# Define the pose of the stereo camera rig in the world frame (left camera's pose).
pose = Pose3(Rot3.Ypr(-np.pi/2, 0, -np.pi/2), Point3(0, 0, 5.0))

# Define the stereo calibration.
fx, fy, s, u0, v0, b = 1000, 1000, 0, 640, 360, 0.5
K = Cal3_S2Stereo(fx, fy, s, u0, v0, b)

# Create a StereoCamera object.
stereo_camera = StereoCamera(pose, K)

stereo_camera.print("My Stereo Camera")

My Stereo Camera.camera. R: [
	6.12323e-17, 6.12323e-17, 1;
	-1, 3.7494e-33, 6.12323e-17;
	-0, -1, 6.12323e-17
]
t: 0 0 5
My Stereo Camera.calibration. K: 1000    0  640
   0 1000  360
   0    0    1
Baseline: 0.5


## Properties

You can access the camera's pose, calibration, and baseline using the following methods:

In [3]:
camera_pose = stereo_camera.pose()
calibration = stereo_camera.calibration()
baseline = stereo_camera.baseline()

print("Camera Pose:")
print(camera_pose)
print("\nCalibration:\n", calibration)
print(f"\nBaseline: {baseline}")

Camera Pose:
R: [
	6.12323e-17, 6.12323e-17, 1;
	-1, 3.7494e-33, 6.12323e-17;
	-0, -1, 6.12323e-17
]
t: 0 0 5


Calibration:
 K: 1000    0  640
   0 1000  360
   0    0    1
Baseline: 0.5


Baseline: 0.5


## Basic Operations

### `project`

The primary function of the `StereoCamera` is to project 3D world points into 2D stereo image coordinates. The `project` method takes a `Point3` and returns a `StereoPoint2`, which contains the pixel coordinates $(u_L, u_R, v)$ for the left and right images.

If a point is behind the camera, a `StereoCheiralityException` is thrown.

In [4]:
# Define a 3D point in the world frame.
p_world = Point3(5, 3, 2)

# Project the 3D point into the stereo camera.
p_stereo = stereo_camera.project(p_world)

print(f"3D Point in world frame: {p_world}")
print(f"Projected StereoPoint2 (uL, uR, v): {p_stereo}")

3D Point in world frame: [5. 3. 2.]
Projected StereoPoint2 (uL, uR, v): (40, -60, 960)



The `project` method can also calculate the Jacobians of the projection with respect to the camera pose (`H1`) and the 3D point (`H2`).

In [6]:
H1 = np.zeros((3, 6), order='F') # Jacobian wrt camera pose
H2 = np.zeros((3, 3), order='F') # Jacobian wrt point

p_stereo = stereo_camera.project([p_world, H1, H2])

print("Jacobian wrt Pose (H1):\n", H1)
print("\nJacobian wrt Point (H2):\n", H2)

TypeError: project(): incompatible function arguments. The following argument types are supported:
    1. (self: gtsam.gtsam.StereoCamera, point: numpy.ndarray[numpy.float64[3, 1]]) -> gtsam.gtsam.StereoPoint2

Invoked with: .camera. R: [
	6.12323e-17, 6.12323e-17, 1;
	-1, 3.7494e-33, 6.12323e-17;
	-0, -1, 6.12323e-17
]
t: 0 0 5
.calibration. K: 1000    0  640
   0 1000  360
   0    0    1
Baseline: 0.5
, [array([5., 3., 2.]), array([[0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.]]), array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.]])]

### `backproject`

The `backproject` method performs the inverse operation, reconstructing a 3D `Point3` from a `StereoPoint2`.

In [ ]:
reprojected_point = stereo_camera.backproject(p_stereo)

print(f"Original Point: {p_world}")
print(f"Reprojected Point: {reprojected_point}")

## Manifold Operations

`StereoCamera` is also a manifold, which allows it to be used in optimization problems. The manifold operations `retract` and `localCoordinates` are applied to the `Pose3` component of the camera, while the calibration is held constant.

In [ ]:
print("Original StereoCamera:")
stereo_camera.print()

# Define a delta vector in the tangent space.
delta = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6])

# Retract the camera pose.
retracted_camera = stereo_camera.retract(delta)
print("\nRetracted StereoCamera:")
retracted_camera.print()

# Calculate the local coordinates between the original and retracted cameras.
local_coords = stereo_camera.localCoordinates(retracted_camera)
print("\nLocal Coordinates:", local_coords)

## Additional Resources

The following resources provide more detailed explanations of camera calibration and stereo vision.

### Article(s)
- ["5.3. Cameras for Robot Vision"](https://www.roboticsbook.org/S53_diffdrive_sensing.html) by Dr. Frank Dellaert and Dr. Seth Hutchinson

### Video(s)
- ["Simple Stereo | Camera Calibration"](https://www.youtube.com/watch?v=hUVyD4z2-eY) by Dr. Shree Nayar from Columbia University

## Source
- [StereoCamera.h](https://github.com/borglab/gtsam/blob/develop/gtsam/geometry/StereoCamera.h)
- [StereoCamera.cpp](https://github.com/borglab/gtsam/blob/develop/gtsam/geometry/StereoCamera.cpp)